# CIFAR-100 Retrieval Results

This notebook visualizes retrieval metrics produced by `run_cifar100_retrieval_metrics.sh`.

Expected run output:

- `cifar100_retrieval_summary.csv`
- `retrieval_metrics/*.json`

The plots compare methods across representation dimension and retrieval shortlist size `k`.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()

# Set this to a concrete path to inspect a specific experiment.
# Example: EXPERIMENT_DIR = ROOT / "cifar100_runs" / "cifar100_seed_0_20260601_234841"
EXPERIMENT_DIR = None

# Used by all focused plots below. Change these two values and re-run the plotting cells.
SELECTED_K = 10
SELECTED_METRIC = "mAP"  # one of: mAP, precision, recall, topk

PERCENT_METRICS = ["top1", "mAP", "precision", "recall", "topk"]
METRIC_LABELS = {
    "top1": "Top-1 retrieval accuracy",
    "mAP": "mAP@k",
    "precision": "Precision@k",
    "recall": "Recall@k",
    "topk": "Top-k retrieval accuracy",
}

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

In [ ]:
def latest_retrieval_run(root):
    runs_root = root / "cifar100_runs"
    candidates = []
    if runs_root.exists():
        for path in runs_root.glob("cifar100_seed_*"):
            if not path.is_dir():
                continue
            if (path / "cifar100_retrieval_summary.csv").exists() or (path / "retrieval_metrics").exists():
                candidates.append(path)
    if not candidates:
        raise FileNotFoundError("No CIFAR-100 retrieval outputs found under cifar100_runs/.")
    return sorted(candidates, key=lambda path: path.stat().st_mtime)[-1]

if EXPERIMENT_DIR is None:
    EXPERIMENT_DIR = latest_retrieval_run(ROOT)
else:
    EXPERIMENT_DIR = Path(EXPERIMENT_DIR)

SUMMARY_CSV = EXPERIMENT_DIR / "cifar100_retrieval_summary.csv"
METRICS_DIR = EXPERIMENT_DIR / "retrieval_metrics"

print(f"Using experiment: {EXPERIMENT_DIR}")
print(f"Summary CSV: {SUMMARY_CSV if SUMMARY_CSV.exists() else 'not found'}")
print(f"Metrics JSON dir: {METRICS_DIR if METRICS_DIR.exists() else 'not found'}")

In [ ]:
METHOD_ORDER = [
    "MRL",
    "MRL-E",
    "BOR-MRL matrix_exp",
    "Independent-block BOR-MRL",
    "BOR-MRL frozen",
    "BOR-MRL cayley",
    "BOR-MRL householder",
    "Full feature",
    "Fixed 512",
]
METHOD_RANK = {method: idx for idx, method in enumerate(METHOD_ORDER)}

RUN_LABELS = {
    "mrl": "MRL",
    "mrle": "MRL-E",
    "bor_mrl": "BOR-MRL matrix_exp",
    "bor_block_mrl": "Independent-block BOR-MRL",
    "bor_mrl_frozen": "BOR-MRL frozen",
    "bor_mrl_cayley": "BOR-MRL cayley",
    "bor_mrl_householder": "BOR-MRL householder",
    "full_feature": "Full feature",
}

def method_label(method_name, row=None):
    method_name = str(method_name)
    if method_name in RUN_LABELS:
        return RUN_LABELS[method_name]
    if method_name.startswith("fixed_"):
        return f"Fixed {method_name.split('_', 1)[1]}"
    if method_name == "bor_mrl_identity":
        return "BOR-MRL identity (old)"
    return method_name.replace("_", " ")

def method_sort_key(method):
    return METHOD_RANK.get(method, len(METHOD_RANK))

def load_from_json(metrics_dir):
    rows = []
    if not metrics_dir.exists():
        return pd.DataFrame()
    for path in sorted(metrics_dir.glob("*.json")):
        with open(path) as handle:
            payload = json.load(handle)
        for metric in payload.get("metrics", []):
            rows.append({
                "method": path.stem,
                "model": payload.get("model"),
                "feature_config": payload.get("feature_config"),
                "eval_config": payload.get("eval_config"),
                "index_type": payload.get("index_type"),
                "dim": metric.get("dim"),
                "k": metric.get("k"),
                "top1": metric.get("top1"),
                "mAP": metric.get("mAP"),
                "precision": metric.get("precision"),
                "recall": metric.get("recall"),
                "topk": metric.get("topk"),
                "neighbors_path": metric.get("neighbors_path"),
            })
    return pd.DataFrame(rows)

if SUMMARY_CSV.exists():
    retrieval_df = pd.read_csv(SUMMARY_CSV)
else:
    retrieval_df = load_from_json(METRICS_DIR)

if retrieval_df.empty:
    raise FileNotFoundError(f"No retrieval metrics found in {EXPERIMENT_DIR}")

for column in ["dim", "k", *PERCENT_METRICS]:
    retrieval_df[column] = pd.to_numeric(retrieval_df[column], errors="coerce")

retrieval_df = retrieval_df.dropna(subset=["method", "dim", "k", "mAP"]).copy()
retrieval_df["dim"] = retrieval_df["dim"].astype(int)
retrieval_df["k"] = retrieval_df["k"].astype(int)
retrieval_df["method_label"] = retrieval_df["method"].map(method_label)
retrieval_df["method_rank"] = retrieval_df["method_label"].map(method_sort_key)

for metric in PERCENT_METRICS:
    retrieval_df[f"{metric}_pct"] = retrieval_df[metric] * 100

retrieval_df = retrieval_df.sort_values(["method_rank", "method_label", "dim", "k"]).reset_index(drop=True)
retrieval_df.head()

## What The Metrics Mean

- `top1`: first retrieved neighbor has the same CIFAR-100 label as the query.
- `topk`: at least one of the first `k` neighbors has the same label as the query.
- `precision@k`: fraction of the first `k` neighbors with the query label.
- `recall@k`: fraction of all database images with the query label that were retrieved. For CIFAR-100 this denominator is usually 500.
- `mAP@k`: rank-sensitive average precision over the first `k` neighbors. Higher means correct-class neighbors appear earlier.

In [ ]:
overview = (
    retrieval_df.groupby("method_label")
    .agg(
        dims=("dim", lambda values: ", ".join(map(str, sorted(values.unique())))),
        ks=("k", lambda values: ", ".join(map(str, sorted(values.unique())))),
        rows=("method_label", "size"),
    )
    .reset_index()
    .sort_values("method_label", key=lambda series: series.map(method_sort_key))
)

print(f"Rows: {len(retrieval_df):,}")
print(f"Methods: {retrieval_df['method_label'].nunique()}")
print(f"Dimensions: {sorted(retrieval_df['dim'].unique())}")
print(f"k values: {sorted(retrieval_df['k'].unique())}")
overview

## Top-1 Retrieval Across Dimensions

`top1` is independent of the metric `k`, so the plot below deduplicates rows by method and dimension.

In [ ]:
top1_df = retrieval_df.drop_duplicates(["method_label", "dim"])

fig, ax = plt.subplots(figsize=(11, 6))
for method in sorted(top1_df["method_label"].unique(), key=method_sort_key):
    group = top1_df[top1_df["method_label"] == method].sort_values("dim")
    ax.plot(group["dim"], group["top1_pct"], marker="o", linewidth=2, label=method)

ax.set_xscale("log", base=2)
ax.set_xlabel("Representation dimension")
ax.set_ylabel("Top-1 retrieval accuracy (%)")
ax.set_title("Nearest-neighbor retrieval accuracy by representation size")
ax.legend(loc="best")
plt.tight_layout()

## Metric vs Dimension At A Fixed k

Change `SELECTED_K` and `SELECTED_METRIC` in the first code cell to focus this view.

In [ ]:
available_k = sorted(retrieval_df["k"].unique())
if SELECTED_K not in available_k:
    SELECTED_K = available_k[0]
if SELECTED_METRIC not in ["mAP", "precision", "recall", "topk"]:
    raise ValueError("SELECTED_METRIC must be one of: mAP, precision, recall, topk")

focus_df = retrieval_df[retrieval_df["k"] == SELECTED_K].copy()

fig, ax = plt.subplots(figsize=(11, 6))
for method in sorted(focus_df["method_label"].unique(), key=method_sort_key):
    group = focus_df[focus_df["method_label"] == method].sort_values("dim")
    ax.plot(group["dim"], group[f"{SELECTED_METRIC}_pct"], marker="o", linewidth=2, label=method)

ax.set_xscale("log", base=2)
ax.set_xlabel("Representation dimension")
ax.set_ylabel(f"{METRIC_LABELS[SELECTED_METRIC]} (%)")
ax.set_title(f"{METRIC_LABELS[SELECTED_METRIC]} at k={SELECTED_K}")
ax.legend(loc="best")
plt.tight_layout()

## mAP@k Curves

This view shows how quickly each method improves as the neighbor list gets longer. Each method is evaluated at its best dimension for the selected `k` values.

In [ ]:
best_map_by_method_k = (
    retrieval_df.sort_values(["method_rank", "method_label", "k", "mAP"], ascending=[True, True, True, False])
    .groupby(["method_label", "k"], as_index=False)
    .first()
)

fig, ax = plt.subplots(figsize=(11, 6))
for method in sorted(best_map_by_method_k["method_label"].unique(), key=method_sort_key):
    group = best_map_by_method_k[best_map_by_method_k["method_label"] == method].sort_values("k")
    ax.plot(group["k"], group["mAP_pct"], marker="o", linewidth=2, label=method)

ax.set_xscale("log", base=2)
ax.set_xlabel("k")
ax.set_ylabel("Best mAP@k (%)")
ax.set_title("Best mAP over representation dimensions")
ax.legend(loc="best")
plt.tight_layout()

## Heatmap At The Selected k

Rows are methods and columns are dimensions. This makes it easy to spot whether a method needs the full representation or performs well at small dimensions.

In [ ]:
heatmap_df = focus_df.pivot_table(
    index="method_label",
    columns="dim",
    values=f"{SELECTED_METRIC}_pct",
    aggfunc="max",
)
heatmap_df = heatmap_df.reindex(sorted(heatmap_df.index, key=method_sort_key))
heatmap_df = heatmap_df.reindex(sorted(heatmap_df.columns), axis=1)

fig, ax = plt.subplots(figsize=(12, max(4, 0.55 * len(heatmap_df))))
image = ax.imshow(heatmap_df.to_numpy(), aspect="auto", cmap="viridis")

ax.set_xticks(np.arange(len(heatmap_df.columns)))
ax.set_xticklabels(heatmap_df.columns)
ax.set_yticks(np.arange(len(heatmap_df.index)))
ax.set_yticklabels(heatmap_df.index)
ax.set_xlabel("Representation dimension")
ax.set_title(f"{METRIC_LABELS[SELECTED_METRIC]} at k={SELECTED_K} (%)")

for row_idx in range(heatmap_df.shape[0]):
    for col_idx in range(heatmap_df.shape[1]):
        value = heatmap_df.iloc[row_idx, col_idx]
        if pd.notna(value):
            ax.text(col_idx, row_idx, f"{value:.1f}", ha="center", va="center", color="white", fontsize=8)

fig.colorbar(image, ax=ax, label="Percent")
plt.tight_layout()

## Best Configuration Per Method

The table below picks the best dimension per method at the selected `k` and metric.

In [ ]:
best_focus = (
    focus_df.sort_values([f"{SELECTED_METRIC}_pct", "dim"], ascending=[False, True])
    .groupby("method_label", as_index=False)
    .first()
    .sort_values([f"{SELECTED_METRIC}_pct", "top1_pct"], ascending=[False, False])
)

best_table = best_focus[[
    "method_label", "dim", "top1_pct", "mAP_pct", "precision_pct", "recall_pct", "topk_pct",
]].rename(columns={
    "method_label": "method",
    "top1_pct": "top1 (%)",
    "mAP_pct": "mAP (%)",
    "precision_pct": "precision (%)",
    "recall_pct": "recall (%)",
    "topk_pct": "top-k (%)",
})

best_table.style.format({
    "top1 (%)": "{:.2f}",
    "mAP (%)": "{:.2f}",
    "precision (%)": "{:.2f}",
    "recall (%)": "{:.2f}",
    "top-k (%)": "{:.2f}",
})

## Smallest Dimension Near Each Method's Best

This table asks a practical question: what is the smallest dimension that gets within 95% of each method's best selected metric at the selected `k`?

In [ ]:
threshold_rows = []
for method, group in focus_df.groupby("method_label"):
    group = group.sort_values("dim")
    best_value = group[f"{SELECTED_METRIC}_pct"].max()
    eligible = group[group[f"{SELECTED_METRIC}_pct"] >= 0.95 * best_value]
    chosen = eligible.iloc[0]
    best_row = group.loc[group[f"{SELECTED_METRIC}_pct"].idxmax()]
    threshold_rows.append({
        "method": method,
        "smallest_dim_95pct_best": int(chosen["dim"]),
        f"{SELECTED_METRIC}_at_smallest_dim (%)": chosen[f"{SELECTED_METRIC}_pct"],
        "best_dim": int(best_row["dim"]),
        f"best_{SELECTED_METRIC} (%)": best_value,
    })

threshold_df = pd.DataFrame(threshold_rows).sort_values(
    [f"best_{SELECTED_METRIC} (%)", "smallest_dim_95pct_best"],
    ascending=[False, True],
)
threshold_df.style.format({
    f"{SELECTED_METRIC}_at_smallest_dim (%)": "{:.2f}",
    f"best_{SELECTED_METRIC} (%)": "{:.2f}",
})

## All Raw Metrics

Use this final table for inspection, filtering, and export.

In [ ]:
display_columns = [
    "method_label", "dim", "k", "top1_pct", "mAP_pct", "precision_pct", "recall_pct", "topk_pct",
    "index_type", "feature_config", "neighbors_path",
]

raw_table = retrieval_df[display_columns].rename(columns={
    "method_label": "method",
    "top1_pct": "top1 (%)",
    "mAP_pct": "mAP (%)",
    "precision_pct": "precision (%)",
    "recall_pct": "recall (%)",
    "topk_pct": "top-k (%)",
})

raw_table.style.format({
    "top1 (%)": "{:.2f}",
    "mAP (%)": "{:.2f}",
    "precision (%)": "{:.2f}",
    "recall (%)": "{:.2f}",
    "top-k (%)": "{:.2f}",
})